# IQA-RAG: Exploratory Research & Prototyping Notebook

This notebook demonstrates the end-to-end components of the **IQA-RAG** system:
1. **Ingestion & Fetching**: Inspecting downloaded arXiv papers
2. **Parsing & Cleaning**: Extracting clean text and detecting academic section boundaries
3. **Section-Aware Chunking**: Slicing documents into granular, non-overlapping semantic chunks
4. **Vector Indexing & Similarity**: Generating dense embeddings and querying the vector store
5. **Grounded Generation**: Synthesizing answers with source citations using LLMs
6. **Evaluation & Faithfulness**: Calculating Hit-Rate@K and claim verification

In [1]:
import sys
import json
from pathlib import Path

# Ensure project root in sys.path
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import default_config
from src.ingest.download import ArxivDownloader
from src.ingest.parse import PDFParser
from src.ingest.chunk import SectionAwareChunker
from src.index.vectorstore import IQAVectorStore
from src.pipeline.rag import IQARAGPipeline
from src.eval.metrics import calculate_hit_rate_at_k, evaluate_faithfulness

print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {default_config.data_dir}")

## 1. Inspecting Ingested Chunks

Let's inspect the chunk distribution across different IQA papers in `data/processed/chunks.json`.

In [2]:
chunks_file = default_config.processed_dir / "chunks.json"
with open(chunks_file, "r", encoding="utf-8") as f:
    chunks = json.load(f)

print(f"Total chunks available: {len(chunks)}")
papers = set(c["paper_title"] for c in chunks)
print(f"Unique papers ({len(papers)}):")
for p in sorted(papers):
    p_chunks = [c for c in chunks if c["paper_title"] == p]
    avg_tokens = sum(c["token_count"] for c in p_chunks) / len(p_chunks)
    print(f" - {p[:65]:<65} | Chunks: {len(p_chunks):<3} | Avg Tokens: {avg_tokens:.1f}")

## 2. Testing Semantic Vector Retrieval

Query the vector store with research questions to inspect similarity scores and retrieved sections.

In [3]:
store = IQAVectorStore()
query = "How does BRISQUE compute quality features in the spatial domain?"
results = store.query(query, top_k=3)

print(f"Query: {query}\n" + "="*60)
for idx, r in enumerate(results, start=1):
    meta = r["metadata"]
    print(f"[{idx}] Score: {r['score']} | Paper: {meta.get('paper_title')} | Section: {meta.get('section_title')}")
    print(f"    Excerpt: {r['document'][:140]}...\n")

## 3. End-to-End RAG Pipeline Execution

Run full question answering with grounded citations.

In [4]:
pipeline = IQARAGPipeline()
q = "What key hypothesis does LPIPS investigate regarding deep neural network representations?"
res = pipeline.run(q, top_k=3)

print("QUESTION:", res["query"])
print("MODEL:", res["model"])
print("LATENCY:", res["total_latency_ms"], "ms")
print("CITATIONS:", res["citations"])
print("\nGROUNDED ANSWER:")
print(res["answer"])

## 4. Metric Evaluation on a Sample Query

Calculate Hit-Rate@K and Faithfulness score.

In [5]:
eval_set_path = default_config.eval_set_path
with open(eval_set_path, "r", encoding="utf-8") as f:
    eval_set = json.load(f)

sample_item = eval_set[2]  # LPIPS question
print(f"Evaluating ID: {sample_item['id']} - {sample_item['question']}")

retrieved = res["retrieved_docs"]
hr1 = calculate_hit_rate_at_k(retrieved, sample_item["target_paper"], sample_item["target_section"], sample_item["keywords"], k=1)
hr3 = calculate_hit_rate_at_k(retrieved, sample_item["target_paper"], sample_item["target_section"], sample_item["keywords"], k=3)
faith = evaluate_faithfulness(res["answer"], retrieved)

print(f"Hit-Rate @ 1 : {hr1}")
print(f"Hit-Rate @ 3 : {hr3}")
print(f"Faithfulness : {faith}")